# Task 035: mne-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: MEG/EEG source localization using minimum-norm estimation

**Paper**: MEG and EEG data analysis with MNE-Python
**Repository**: https://github.com/mne-tools/mne-python

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 220.63 dB
- **SSIM**: N/A


### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt
import os
import mne
from mne.datasets import sample

# -----------------------------------------------------------------------------
# 1. Data Loading and Preprocessing
# -----------------------------------------------------------------------------

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
# -----------------------------------------------------------------------------
# 1. Data Loading and Preprocessing
# -----------------------------------------------------------------------------
def load_and_preprocess_data():
    """
    Loads sample data, computes noise covariance, and prepares matrices for inversion.
    
    Returns:
        tuple: (G, C, y, nave, P, info, evoked_ref)
            - G (np.ndarray): Gain matrix (n_channels, n_sources)
            - C (np.ndarray): Noise covariance (n_channels, n_channels)
            - y (np.ndarray): Sensor data (n_channels, n_times)
            - nave (int): Number of averages
            - P (np.ndarray): Projection matrix (n_channels, n_channels)
            - info (mne.Info): Measurement info
            - evoked_ref (mne.Evoked): Reference evoked object for MNE comparison
    """
    print("\n=== Phase 1: Data Preprocessing & Forward Model (MNE) ===")
    data_path = sample.data_path()
    raw_fname = os.path.join(data_path, 'MEG', 'sample', 'sample_audvis_filt-0-40_raw.fif')
    event_fname = os.path.join(data_path, 'MEG', 'sample', 'sample_audvis_filt-0-40_raw-eve.fif')
    fwd_fname = os.path.join(data_path, 'MEG', 'sample', 'sample_audvis-meg-eeg-oct-6-fwd.fif')
    
    # Read Raw
    raw = mne.io.read_raw_fif(raw_fname, preload=True)
    events = mne.read_events(event_fname)
    
    # Pick MEG channels
    picks = mne.pick_types(raw.info, meg=True, eeg=False, eog=True, exclude='bads')
    
    # Create Epochs (Left Auditory)
    event_id, tmin, tmax = 1, -0.2, 0.5
    epochs = mne.Epochs(raw, events, event_id, tmin, tmax, picks=picks,
                        baseline=(None, 0), reject=dict(grad=4000e-13, mag=4e-12, eog=150e-6))
    
    # Average epochs (removed unsupported 'verbose' kwarg)
    evoked = epochs.average()
    
    # Compute Noise Covariance
    noise_cov = mne.compute_covariance(epochs, tmax=0, method=['shrunk', 'empirical'])
    
    # Read Forward Solution
    forward = mne.read_forward_solution(fwd_fname)
    
    # Convert to FIXED orientation (simplifies the problem to 1 source per vertex)
    forward = mne.convert_forward_solution(forward, surf_ori=True, force_fixed=True, use_cps=True)
    forward = mne.pick_types_forward(forward, meg=True, eeg=False)
    
    # Ensure evoked picks match forward
    evoked.pick_types(meg=True, eeg=False)
    
    # Intersect channels between Info, Forward, and Covariance
    info = evoked.info
    common_channels = [ch for ch in info['ch_names'] 
                       if ch in forward['info']['ch_names'] 
                       and ch in noise_cov['names']]
    print(f"Number of common channels: {len(common_channels)}")
    
    # Subset and reorder
    evoked.pick_channels(common_channels)
    forward = mne.pick_channels_forward(forward, common_channels)
    noise_cov = mne.pick_channels_cov(noise_cov, common_channels)
    
    # Extract numpy arrays
    y = evoked.data  # (n_channels, n_times)
    G = forward['sol']['data']  # (n_channels, n_sources)
    C = noise_cov['data']  # (n_channels, n_channels)
    nave = evoked.nave
    
    # --- Compute Projection Matrix P ---
    def compute_proj_matrix(projs, ch_names):
        n_ch = len(ch_names)
        P_out = np.eye(n_ch)
        all_vectors = []
        for p in projs:
            if p['active']:
                proj_ch_names = p['data']['col_names']
                vecs = p['data']['data']
                for v in vecs:
                    full_v = np.zeros(n_ch)
                    for i, ch in enumerate(proj_ch_names):
                        if ch in ch_names:
                            idx = ch_names.index(ch)
                            full_v[idx] = v[i]
                    all_vectors.append(full_v)
        
        if not all_vectors:
            return P_out
            
        U_mat = np.array(all_vectors).T
        Q, _ = scipy.linalg.qr(U_mat, mode='economic')
        P_out = np.eye(n_ch) - np.dot(Q, Q.T)
        return P_out

    P = compute_proj_matrix(info['projs'], common_channels)
    
    # Also return the Forward object and NoiseCov object for the reference MNE run
    # Pack these into a wrapper or simple struct if needed, but here we just attach 
    # them to the evoked object or return them separately.
    # We will return 'evoked' (which has info) and 'forward', 'noise_cov' separately 
    # for the evaluation phase.
    
    # To strictly follow the signature requested, we return the matrices for the standalone solver
    # and objects for the MNE reference solver.
    return G, C, y, nave, P, info, evoked, forward, noise_cov

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
# -----------------------------------------------------------------------------
# 2. Forward Operator
# -----------------------------------------------------------------------------
def forward_operator(x, G):
    """
    Computes the forward model y_pred = G * x.
    This simulates sensor data from source currents.
    
    Args:
        x (np.ndarray): Source currents (n_sources, n_times)
        G (np.ndarray): Gain matrix (n_channels, n_sources)
        
    Returns:
        y_pred (np.ndarray): Simulated sensor data (n_channels, n_times)
    """
    # Simple matrix multiplication representing the physical forward model
    y_pred = np.dot(G, x)
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
# -----------------------------------------------------------------------------
# 3. Run Inversion
# -----------------------------------------------------------------------------
def run_inversion(G, C, y, nave, P, snr=3.0, method='dSPM'):
    """
    Computes the inverse operator K and applies it to sensor data y.
    
    Mathematical Steps:
    1. C_scaled = C / nave
    2. Project C and G using P
    3. Whiten G -> G_tilde
    4. SVD of G_tilde
    5. Compute K
    6. Apply K to y
    7. Apply noise normalization (if dSPM)
    
    Returns:
        source_estimate (np.ndarray): (n_sources, n_times)
    """
    print("\n--- [Standalone] Fitting Inverse Model ---")
    lambda2 = 1.0 / (snr ** 2)
    
    # 0. Adjust Noise Covariance for Averaging
    C_scaled = C / nave
    
    # 1. Apply Projections
    # C_proj = P * C_scaled * P^T
    C_proj = np.dot(P, np.dot(C_scaled, P.T))
    # G_proj = P * G
    G_proj = np.dot(P, G)
    
    # 2. Compute Whitener W = C^(-1/2) using eigh
    eig_vals, eig_vecs = scipy.linalg.eigh(C_proj)
    
    # Filter small eigenvalues
    max_eig = np.max(eig_vals)
    tol = 1e-6 * max_eig
    mask = eig_vals > tol
    
    print(f"Rank of Noise Covariance: {np.sum(mask)} / {len(eig_vals)}")
    
    eig_vals = eig_vals[mask]
    eig_vecs = eig_vecs[:, mask]
    
    # W = Lambda^(-1/2) V^T
    W = np.dot(eig_vecs, np.dot(np.diag(1.0 / np.sqrt(eig_vals)), eig_vecs.T))
    
    # 3. Whiten the Gain Matrix: G_tilde = W * G_proj
    G_tilde = np.dot(W, G_proj)
    
    # --- MNE Scaling Step ---
    n_nzero = np.sum(mask)
    trace_GRGT = np.sum(G_tilde ** 2)
    g_scale = np.sqrt(n_nzero / trace_GRGT)
    print(f"Gain scaling factor: {g_scale:.4e}")
    
    G_tilde = G_tilde * g_scale
    
    # 4. SVD of G_tilde = U * S * V^T
    U, S, Vh = scipy.linalg.svd(G_tilde, full_matrices=False)
    V = Vh.T
    
    # 5. Compute Inverse Operator Weights
    # Gamma = S / (S^2 + lambda2)
    S2 = S ** 2
    Gamma_diag = S / (S2 + lambda2)
    
    # 6. Assemble K
    # K = (V * Gamma) * U^T * W
    KV = V * Gamma_diag  # Broadcast
    KU = np.dot(U.T, W)
    K = np.dot(KV, KU)
    
    # Apply source covariance scaling
    K *= g_scale
    
    # 7. Apply K to data
    print(f"--- [Standalone] Applying {method} Inverse ---")
    source_estimate = np.dot(K, y)
    
    # 8. Apply dSPM normalization if requested
    if method == 'dSPM':
        # noise_var = diag( V * Gamma^2 * V^T ) * g_scale^2
        noise_var = np.sum((V * Gamma_diag) ** 2, axis=1) * (g_scale ** 2)
        noise_norm = 1.0 / np.sqrt(noise_var)
        source_estimate *= noise_norm[:, np.newaxis]
        
    return source_estimate

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
# -----------------------------------------------------------------------------
# 4. Evaluate Results
# -----------------------------------------------------------------------------
def evaluate_results(x_hat, info, evoked, forward, noise_cov):
    """
    Compares standalone results against the reference MNE implementation.
    Generates metrics and plots.
    """
    print("\n=== Phase 3: Reference MNE Reconstruction ===")
    
    # Create inverse operator using MNE (Reference)
    inv_mne = mne.minimum_norm.make_inverse_operator(
        info, forward, noise_cov, 
        loose=0.0, depth=None, fixed=True, verbose=False
    )
    
    # Apply MNE inverse
    stc_mne = mne.minimum_norm.apply_inverse(
        evoked, inv_mne, lambda2=1.0/9.0, method='dSPM', verbose=False
    )
    
    x_mne = stc_mne.data
    
    print("\n=== Phase 4: Evaluation ===")
    print(f"Standalone shape: {x_hat.shape}")
    print(f"MNE shape: {x_mne.shape}")
    
    # Compute Metrics
    mse = np.mean((x_hat - x_mne) ** 2)
    if mse == 0:
        psnr = np.inf
    else:
        psnr = 10 * np.log10(np.max(x_mne)**2 / mse)
    
    corr = np.corrcoef(x_hat.ravel(), x_mne.ravel())[0, 1]
    
    print(f"MSE between Standalone and MNE: {mse:.6e}")
    print(f"PSNR: {psnr:.2f} dB")
    print(f"Correlation: {corr:.6f}")
    
    if corr > 0.99:
        print("SUCCESS: Standalone implementation matches MNE reference!")
    else:
        print("WARNING: Discrepancy detected.")
        
    # Visualization
    max_idx = np.argmax(np.sum(x_mne**2, axis=1))
    
    plt.figure(figsize=(10, 5))
    plt.plot(evoked.times, x_mne[max_idx], label='MNE Reference', linewidth=2)
    plt.plot(evoked.times, x_hat[max_idx], '--', label='Standalone', linewidth=2)
    plt.title(f'Source Time Course (Vertex {max_idx})')
    plt.xlabel('Time (s)')
    plt.ylabel('dSPM value')
    plt.legend()
    plt.grid(True)
    output_img = 'comparison_plot.png'
    plt.savefig(output_img)
    print(f"Comparison plot saved to {output_img}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
G, C, y, nave, P, info, evoked, fwd_obj, cov_obj = load_and_preprocess_data()

### 2. Run Inversion (Standalone)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Run Inversion (Standalone)
x_hat = run_inversion(G, C, y, nave, P, snr=3.0, method='dSPM')

# (Optional) Verify Forward Operator
# We can check if projecting the estimated source back to sensor space 
# roughly matches the data (though dSPM scales things, so magnitude will differ)
# y_reconstructed = forward_operator(x_hat, G)

### 3. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Evaluate
evaluate_results(x_hat, info, evoked, fwd_obj, cov_obj)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **mne-master**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=220.63 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: MEG and EEG data analysis with MNE-Python
- Repository: https://github.com/mne-tools/mne-python
